IPYNB file to create a link of the cases NOT used in the training

In [7]:
DEFAULT_SUMMARY_PATH = "/scratch4/workspace/f007g3j_dartmouth_edu-simple/nnUNet_data/nnUNet_results/{input_dataset}/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/{fold}/{validation}/summary.json"

def read_to_get_num_of_files(dataset, fold):
    path = DEFAULT_SUMMARY_PATH.replace("{validation}", "").replace("summary.json", "").format(input_dataset=dataset, fold=fold)
    
    training_file_path = [f for f in os.listdir(path) if f.startswith("training")]
    print(f"Reading from {path} and found a training file")

    for file in training_file_path:
        with open(os.path.join(path, file), "r") as f:
            lines = f.readlines()
            for line in lines:
                if "This split has" in line:
                    return int(line.strip().split("has")[-1].split("training")[0].strip())
                    break



In [19]:
import os 
import json

# First let's get the fold that we want to do it to and the Datset of course
Dataset = "Dataset111_AutoPet"
fold = 8
splits = f'{os.environ["nnUNet_preprocessed"]}/{Dataset}/splits_final.json'
with open(splits, "r") as f:
    splits_dict = json.load(f)

all_training_cases_splits = f'{os.environ["nnUNet_preprocessed"]}/Dataset999_AutoPet/splits_final.json'
with open(all_training_cases_splits, "r") as f:
    all_training_cases_splits_dict = json.load(f)
all_training_cases = all_training_cases_splits_dict[-1]['train']

# Assertion to make sure it's correct
assert len(all_training_cases) == 1043

print("Generating pseudo labels for fold {fold}")
print(f"This fold has {len(splits_dict[fold]['train'])} training cases")
# read_to_get_num_of_files(Dataset, f'fold_{fold}')

cases_to_generate_pseudo_labels_for = set(all_training_cases) - set(splits_dict[fold]['train'])
print(f"Number of cases to generate pseudo labels for: {len(cases_to_generate_pseudo_labels_for)}")

Generating pseudo labels for fold {fold}
This fold has 730 training cases
Number of cases to generate pseudo labels for: 313


In [40]:
# export cases to generate_pseudo_labels_for to a txt file
import pandas as pd 
pd.DataFrame({"cases_to_generate_pseudo_labels_for": list(cases_to_generate_pseudo_labels_for)}).to_csv(f"cases_to_generate_pseudo_labels_for_fold_{fold}.txt", index=False)

In [37]:
# create a link to this in the Dataset that this belongs in 

import glob 

path_to_generate_pl = f'{os.environ["nnUNet_preprocessed"]}/{Dataset}/train_link_{len(cases_to_generate_pseudo_labels_for)}_cases/'
os.makedirs(path_to_generate_pl, exist_ok=True)

nnUNetPlans_3d_fullres_path  = f'{os.environ["nnUNet_preprocessed"]}/Dataset999_AutoPet/nnUNetPlans_3d_fullres/'
paths_in_nnUNetPlans = glob.glob(os.path.join(nnUNetPlans_3d_fullres_path, "*.pkl"))
assert len(paths_in_nnUNetPlans) == 1611

counter = 0
for path in paths_in_nnUNetPlans:
    case_name = os.path.basename(path).replace(".pkl", "")
    if case_name in cases_to_generate_pseudo_labels_for:
        path_to_link = os.path.join(path_to_generate_pl, os.path.basename(path))
        if not os.path.exists(path_to_link):
            # symlink everything that starts with case_name
            things_to_link = glob.glob(os.path.join(nnUNetPlans_3d_fullres_path, f"{case_name}*"))
            for thing in things_to_link:
                os.symlink(thing, os.path.join(path_to_generate_pl, os.path.basename(thing)))
            counter += 1
print(counter)

313
